# 14. Ética, equidad, privacidad y seguridad

**Fases del guía metodológica cubiertas: 20 (Ética, equidad, privacidad, seguridad y cumplimiento)**



## 20.1 Ética y equidad

- **Grupos afectados**: estudiantes de secundaria (menores). El sistema apoya a
  orientadores; **no** es un diagnóstico médico ni una sanción.
- **Representatividad**: el dataset cubre 2 escuelas portuguesas de un curso; no es
  representativo de otras poblaciones -> solo uso preventivo local.
- **Variables sensibles/proxy**: `sex`, `age` y `romantic` se incluyen como features
  (poder predictivo), pero se auditan métricas por subgrupo y se revisa su peso SHAP.
- **Impacto de errores**: FN = alumno de riesgo sin intervención (coste 2); FP = intervención
  innecesaria (coste 1). La matriz de costes refleja esta asimetría.
- **Supervisión humana**: las alertas requieren validación del orientador (abstención en
  zona de baja confianza).

### 20.1.1 Preparación y predicciones

Cargamos el pipeline final y generamos las probabilidades sobre test (mismas que en la
fase 17). Con ellas mediremos la equidad por sexo y escuela usando el umbral congelado.


In [1]:

import sys, pathlib
ROOT = pathlib.Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import joblib, json, pandas as pd, numpy as np
from src.data.load_data import load_processed
from src.features.build_features import add_domain_features
from src.evaluation.metrics import fairness_by_group
from src.api.main import predict_proba_series

d = load_processed()
Xte = add_domain_features(d["X_test"]); yte = d["y_test"]
meta = json.loads((ROOT / "models" / "final_model_metadata.json").read_text(encoding="utf-8"))
y_proba = predict_proba_series(Xte)
umbral = meta["threshold"]



### 20.1.2 Métricas por subgrupo

Para cada grupo (sexo y escuela) calculamos: tamaño, prevalencia real, tasa de positivos
predicha, accuracy, recall, precision, F1 y ROC-AUC. La comparación entre grupos permite
detectar **sesgos**: si el recall de un grupo es mucho menor, el modelo está fallando más
con esa población (potencial discriminación indirecta).


In [2]:

for grupo in ["sex", "school"]:
    print(f"--- Equidad por {grupo} (test) ---")
    f = fairness_by_group(yte, y_proba, Xte[grupo], threshold=umbral)
    print(f.to_string(index=False))
    print()


--- Equidad por sex (test) ---
group  n  prevalence  predicted_positive_rate  accuracy   recall  precision       f1  roc_auc
    F 77    0.285714                 0.324675  0.753247 0.636364   0.560000 0.595745 0.716529
    M 56    0.535714                 0.660714  0.732143 0.866667   0.702703 0.776119 0.775641

--- Equidad por school (test) ---
group  n  prevalence  predicted_positive_rate  accuracy   recall  precision       f1  roc_auc
   MS 42    0.309524                 0.404762  0.714286 0.692308   0.529412 0.600000 0.713528
   GP 91    0.428571                 0.494505  0.758242 0.794872   0.688889 0.738095 0.789941




### 20.1.3 Criterio de equidad (diferencia de recall)

Aplicamos el criterio de aceptación definido en la fase 6.4: la diferencia de recall
entre sexos debe ser <= 0.20. Si se supera, se documenta como riesgo y se propone
mitigación (recalibración por grupo, revisión de umbrales). Este es un control
cuantitativo, no una opinión: la cifra queda registrada.


In [3]:

# Diferencia de recall entre sexos (criterio de aceptación 6.4: <= 0.2)
f_sex = fairness_by_group(yte, y_proba, Xte["sex"], threshold=umbral)
diff = abs(f_sex.loc[f_sex["group"] == "M", "recall"].values[0] - f_sex.loc[f_sex["group"] == "F", "recall"].values[0])
print(f"Diferencia de recall M-F: {diff:.3f} (criterio <= 0.20 -> {'OK' if diff <= 0.2 else 'REVISAR'})")


Diferencia de recall M-F: 0.230 (criterio <= 0.20 -> REVISAR)



## 20.2 Privacidad

- El dataset es **anónimo**: sin nombres, IDs, direcciones ni datos biométricos.
- No contiene datos de salud clínicos; el consumo es auto-reportado (dato sensible de
  menores -> minimización: solo las variables del estudio).
- **Retención**: los datos crudos viven en `data/raw/` sin exponer; el informe y la API
  devuelven solo la probabilidad/clase, nunca datos personales.
- La API no registra logs con el payload completo (solo métricas de uso anonimizadas).

## 20.3 Seguridad

- **Validación de inputs**: Pydantic (`src/api/main.py`) valida tipos, rangos y categorías
  antes de inferir (fases 19.1/21.3).
- **Secretos**: `.env` en `.gitignore`; no hay claves en el repositorio.
- **Dependencias**: fijadas en `requirements.txt`; actualización controlada.
- **Riesgos del modelo**: no hay LLM ni agentes -> fuera de alcance prompt injection,
  model extraction, etc. El riesgo principal es el **mal uso** (asignar etiquetas
  definitivas a alumnos) -> documentado en la model card (docs/model_card.md).
- **Data poisoning**: el dataset es fijo y auditable (checksum de archivos en README).

### 20.3.1 Representatividad de grupos entre train y test

Comparamos la distribución de `sex`, `school` y `age` entre train y test. Si difieren
mucho, la evaluación en test no representaría a la población de entrenamiento. En nuestro
split estratificado esperamos proporciones muy similares.


In [4]:

# Auditoría de representatividad de grupos (train vs test)
for grupo in ["sex", "school", "age"]:
    tr = d["X_train"][grupo].value_counts(normalize=True).sort_index()
    te = Xte[grupo].value_counts(normalize=True).sort_index()
    print(grupo, "| train:", tr.round(3).to_dict(), "| test:", te.round(3).to_dict())


sex | train: {'F': 0.601, 'M': 0.399} | test: {'F': 0.579, 'M': 0.421}
school | train: {'GP': 0.667, 'MS': 0.333} | test: {'GP': 0.684, 'MS': 0.316}
age | train: {15: 0.174, 16: 0.265, 17: 0.263, 18: 0.232, 19: 0.048, 20: 0.013, 21: 0.003, 22: 0.003} | test: {15: 0.12, 16: 0.248, 17: 0.323, 18: 0.195, 19: 0.113}
